#### Saves 2000 HVG genes and their raw read counts in a matrix

In [1]:
import sys
import os

# Compute project root (go up one level)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

# Add project root to path
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [2]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
sys.path

['/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python310.zip',
 '/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python3.10',
 '/opt/homebrew/Cellar/python@3.10/3.10.17/Frameworks/Python.framework/Versions/3.10/lib/python3.10/lib-dynload',
 '',
 '/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/thesis/code/venvs/scanpy_umap/lib/python3.10/site-packages',
 '/Users/jeromecho/Library/CloudStorage/OneDrive-Personal/Thesis/thesis/code/CellUntangler']

In [4]:
adata = sc.read_h5ad("../../../../data/HGSOC/ALL_CELLS/all_cells_1p.h5ad")

In [5]:
adata = adata.raw.to_adata()

In [6]:
# Save raw counts
adata.layers["counts"] = adata.X.copy()

# Normalize + log for HVG selection
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Compute HVGs
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor="seurat",
    subset=False
)

# Restore raw counts BEFORE slicing
adata.X = adata.layers["counts"]

# Now subset (this subsets X and layers consistently)
adata = adata[:, adata.var["highly_variable"]].copy()

# Optional cleanup
del adata.layers["counts"]


OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [17]:
adata.write("../../../../data/HGSOC/ALL_CELLS/all_cells_1p_hvg2000.h5ad")